In [0]:
-- Player - Close Range Shooting
-- Shots and goals from within 20 feet of the net.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    'MTL' AS team,                     -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,                -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type,                    -- 2 = regular season, 3 = playoffs
    CAST(NULL AS STRING) AS player,    -- e.g. 'Nick Suzuki'; NULL = every player
    CAST(NULL AS STRING) AS position,  -- C, L, R, D, F = all forwards, W = all wingers; NULL = all skaters
    25 AS top_n                        -- rows to show
),
player_games AS (                        -- one row per skater per game dressed (game rosters)
  SELECT r.game_id, r.player_id, p.full_name, r.position_code, r.sweater_number,
         r.team_id, t.abbrev AS team, o.abbrev AS opponent, r.team_id = g.home_team_id AS is_home,
         g.season, g.game_date, g.home_team_id,
         CASE WHEN IF(r.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game   g ON r.game_id   = g.game_id
  JOIN nhl.gold.dim_player p ON r.player_id = p.player_id
  JOIN nhl.gold.dim_team   t ON r.team_id   = t.team_id
  JOIN nhl.gold.dim_team   o ON o.team_id   = IF(r.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
    AND (pr.player IS NULL OR lower(p.full_name) = lower(pr.player))
    AND r.position_code <> 'G'
    AND (pr.position IS NULL OR r.position_code = pr.position
         OR (pr.position = 'F' AND r.position_code IN ('C', 'L', 'R'))
         OR (pr.position = 'W' AND r.position_code IN ('L', 'R')))
),
events AS (                              -- plays where one of those players had a role
  SELECT pg.player_id AS pid,
         IF(f.event_owner_team_id = pg.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength,
         f.*
  FROM nhl.gold.fact_event f
  JOIN player_games pg
    ON f.game_id = pg.game_id
   AND pg.player_id IN (f.scoring_player_id, f.assist1_player_id, f.assist2_player_id,
                        f.shooting_player_id, f.blocking_player_id, f.hitting_player_id,
                        f.hittee_player_id, f.winning_player_id, f.losing_player_id,
                        f.committed_by_player_id, f.drawn_by_player_id, f.player_id)
  WHERE COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
),
shots AS (                               -- every shot attempt by those players
  SELECT pg.full_name, pg.player_id, e.game_id, e.type_desc_key, e.shot_type, e.zone_code,
         e.period_number, e.period_type, e.strength, e.goalie_in_net_id,
         e.type_desc_key = 'goal'                    AS is_goal,
         e.type_desc_key IN ('goal', 'shot-on-goal') AS on_net,
         SQRT(POW(89 - ABS(e.x_coord), 2) + POW(e.y_coord, 2)) AS distance_ft
  FROM events e
  JOIN player_games pg ON pg.player_id = e.pid AND pg.game_id = e.game_id
  WHERE (e.type_desc_key = 'goal' AND e.scoring_player_id = e.pid)
     OR (e.type_desc_key IN ('shot-on-goal', 'missed-shot', 'blocked-shot') AND e.shooting_player_id = e.pid)
)
SELECT * FROM (
  SELECT ROW_NUMBER() OVER (ORDER BY COUNT_IF(on_net AND distance_ft < 20) DESC) AS rank,
         full_name,
         COUNT_IF(on_net AND distance_ft < 20)                             AS close_shots,
         COUNT_IF(is_goal AND distance_ft < 20)                            AS close_goals,
         ROUND(100.0 * COUNT_IF(is_goal AND distance_ft < 20)
                     / NULLIF(COUNT_IF(on_net AND distance_ft < 20), 0), 1) AS close_shooting_pct,
         ROUND(100.0 * COUNT_IF(on_net AND distance_ft < 20)
                     / NULLIF(COUNT_IF(on_net), 0), 1)                      AS share_of_shots_close_pct,
         player_id
  FROM shots
  WHERE zone_code = 'O'
  GROUP BY full_name, player_id
) ranked
WHERE rank <= (SELECT top_n FROM params)
ORDER BY rank;